# PipelineWatch-NG — Module 2: Processing and Feature Engineering (Fixed)
### Sentinel-2 NDVI/NDWI + SAR Change Detection + DBSCAN Clustering

**Fix applied:** matplotlib Agg backend — all plots save to outputs/ without crashing.

---

## Cell 1 — Imports

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import Image, display
import ee, os, json, gc
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.cluster import DBSCAN

GEE_PROJECT_ID = "pipelinewatch-ng"
CACHE_DIR  = "../data/cached"
OUTPUT_DIR = "../outputs"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print("GEE connected: " + str(ee.Number(42).getInfo()))
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)
print("matplotlib backend: " + matplotlib.get_backend())


C:\Users\taylo\anaconda3\envs\pipelinewatch\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


GEE connected: 42
matplotlib backend: Agg


## Cell 2 — Config

In [2]:
ROI_BOUNDS     = [6.50, 5.00, 7.20, 5.80]
ROI_CENTER     = [5.40, 6.85]
ROI_ZOOM       = 9
BASELINE_START = "2023-01-01"
BASELINE_END   = "2023-06-30"
RECENT_START   = "2024-01-01"
RECENT_END     = "2024-06-30"
SAR_SIGMA      = 1.5
CACHE_DIR      = "../data/cached"
OUTPUT_DIR     = "../outputs"
ROI = ee.Geometry.Rectangle(ROI_BOUNDS)
print("Config ready.")


Config ready.


## Cell 3 — Load Module 1 outputs

In [3]:
def load_geojson(filename):
    path = os.path.join(CACHE_DIR, filename)
    if not os.path.exists(path):
        print("  WARNING: " + filename + " not found")
        return None, 0
    with open(path) as f:
        gj = json.load(f)
    n = len(gj.get("features", []))
    print("  " + filename + ": " + str(n) + " features")
    return gj, n

print("Loading Module 1 outputs...")
sar_gj,   n_sar   = load_geojson("sar_dark_spots.geojson")
firms_gj, n_firms = load_geojson("fire_hotspots.geojson")
so2_gj,   n_so2   = load_geojson("so2_anomalies.geojson")

rows = []
if firms_gj and n_firms > 0:
    for feat in firms_gj["features"]:
        coords = feat["geometry"]["coordinates"]
        props  = feat["properties"]
        rows.append({"lon": coords[0], "lat": coords[1],
                     "T21_max_K":    props.get("T21_max_K", 0) or 0,
                     "fire_count":   props.get("fire_count", 0) or 0,
                     "likely_source":props.get("likely_source", "unknown"),
                     "confidence":   props.get("confidence", "low")})
df_firms = pd.DataFrame(rows)
print("FIRMS DataFrame: " + str(len(df_firms)) + " hotspots")
print(df_firms[["lon","lat","T21_max_K","fire_count","likely_source"]].head())


Loading Module 1 outputs...
  sar_dark_spots.geojson: 8 features
  fire_hotspots.geojson: 50 features
  so2_anomalies.geojson: 0 features
FIRMS DataFrame: 50 hotspots
        lon       lat   T21_max_K  fire_count          likely_source
0  6.504926  5.086710  349.299988    2.850496  agricultural_or_other
1  6.508775  5.711601  336.000000    3.562633  agricultural_or_other
2  6.508294  5.795818  338.200012    3.000000  agricultural_or_other
3  6.520085  5.688020  335.399994    3.000000  agricultural_or_other
4  6.518342  5.700798  336.000000    5.846860     refinery_candidate


## Cell 4 — Sentinel-2 and SAR functions

In [4]:
def get_s2_collection(roi, start, end, cloud_pct=30):
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(roi).filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_pct))
        .select(["B2","B3","B4","B8","B11","QA60"])
    )

def mask_s2_clouds(image):
    qa          = image.select("QA60")
    cloud_mask  = qa.bitwiseAnd(1 << 10).eq(0)
    cirrus_mask = qa.bitwiseAnd(1 << 11).eq(0)
    return (image.updateMask(cloud_mask.And(cirrus_mask))
            .divide(10000).copyProperties(image, ["system:time_start"]))

def compute_s2_indices(image):
    nir   = image.select("B8")
    red   = image.select("B4")
    green = image.select("B3")
    swir  = image.select("B11")
    ndvi  = nir.subtract(red).divide(nir.add(red)).rename("NDVI")
    ndwi  = green.subtract(nir).divide(green.add(nir)).rename("NDWI")
    ndmi  = nir.subtract(swir).divide(nir.add(swir)).rename("NDMI")
    return image.addBands(ndvi).addBands(ndwi).addBands(ndmi)

def build_s2_composite(roi, start, end, cloud_pct=30):
    col   = get_s2_collection(roi, start, end, cloud_pct)
    count = col.size().getInfo()
    print("  S2 scenes (" + start + " to " + end + "): " + str(count))
    if count == 0:
        return None, 0
    clean = col.map(mask_s2_clouds).map(compute_s2_indices)
    return clean.median().clip(roi), count

def get_s1_collection(roi, start, end):
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(roi).filterDate(start, end)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING"))
        .select(["VV", "VH"])
    )

def compute_sar_features(image, roi, sigma=1.5):
    vv    = image.select("VV")
    vh    = image.select("VH")
    stats = vv.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=roi, scale=100, maxPixels=1e9, bestEffort=True)
    mean_vv   = ee.Number(stats.get("VV_mean"))
    std_vv    = ee.Number(stats.get("VV_stdDev"))
    threshold = mean_vv.subtract(std_vv.multiply(sigma))
    dark_mask = vv.lt(threshold).rename("dark_spot_mask")
    dark_mag  = vv.subtract(mean_vv).multiply(-1).divide(std_vv).rename("dark_spot_magnitude")
    vv_lin    = ee.Image(10).pow(vv.divide(10))
    vh_lin    = ee.Image(10).pow(vh.divide(10))
    return image.addBands(dark_mask).addBands(dark_mag).addBands(vv_lin.divide(vh_lin).rename("vv_vh_ratio"))

def build_s1_composite(roi, start, end, sigma=1.5):
    col   = get_s1_collection(roi, start, end)
    count = col.size().getInfo()
    print("  S1 scenes (" + start + " to " + end + "): " + str(count))
    if count == 0:
        return None, None, 0
    col = col.map(lambda img: compute_sar_features(img, roi, sigma))
    return col.median().clip(roi), col, count

def compute_sar_change(s1_base, s1_rec, roi, threshold_db=-3.0):
    change        = s1_rec.select("VV").subtract(s1_base.select("VV")).rename("SAR_change_dB")
    new_spill     = change.lt(threshold_db).rename("new_spill_mask")
    chronic_spill = s1_base.select("dark_spot_mask").And(s1_rec.select("dark_spot_mask")).rename("chronic_spill_mask")
    return change.addBands(new_spill).addBands(chronic_spill)

print("All functions defined.")


All functions defined.


## Cell 5 — Run Sentinel-2

In [5]:
print("=== Sentinel-2 ===")
s2_baseline, n_s2_base = build_s2_composite(ROI, BASELINE_START, BASELINE_END)
s2_recent,   n_s2_rec  = build_s2_composite(ROI, RECENT_START,   RECENT_END)

if s2_baseline and s2_recent:
    ndvi_change  = s2_recent.select("NDVI").subtract(s2_baseline.select("NDVI")).rename("NDVI_change")
    veg_loss     = ndvi_change.lt(-0.15).rename("vegetation_loss_mask")
    ndvi_anomaly = ndvi_change.addBands(veg_loss)

    gc.collect()
    base_stats = s2_baseline.select("NDVI").reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=ROI, scale=100, maxPixels=1e9, bestEffort=True).getInfo()
    rec_stats = s2_recent.select("NDVI").reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=ROI, scale=100, maxPixels=1e9, bestEffort=True).getInfo()

    base_mean = base_stats.get("NDVI_mean", 0) or 0
    rec_mean  = rec_stats.get("NDVI_mean",  0) or 0
    delta     = rec_mean - base_mean
    direction = "DECREASE (vegetation loss)" if delta < 0 else "INCREASE (recovery)"
    print("Baseline NDVI mean : " + str(round(base_mean, 3)))
    print("Recent NDVI mean   : " + str(round(rec_mean, 3)))
    print("Change             : " + str(round(delta, 3)) + "  ->  " + direction)
else:
    ndvi_anomaly = None
    print("No S2 composites.")


=== Sentinel-2 ===
  S2 scenes (2023-01-01 to 2023-06-30): 56
  S2 scenes (2024-01-01 to 2024-06-30): 48
Baseline NDVI mean : 0.462
Recent NDVI mean   : 0.509
Change             : 0.047  ->  INCREASE (recovery)


## Cell 6 — Run SAR change detection

In [6]:
print("=== SAR Change Detection ===")
s1_baseline, _, n_s1_base = build_s1_composite(ROI, BASELINE_START, BASELINE_END)
s1_recent,   _, n_s1_rec  = build_s1_composite(ROI, RECENT_START,   RECENT_END)
sar_change = compute_sar_change(s1_baseline, s1_recent, ROI)

gc.collect()
change_stats = sar_change.select("SAR_change_dB").reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True)
                            .combine(ee.Reducer.minMax(), sharedInputs=True),
    geometry=ROI, scale=100, maxPixels=1e9, bestEffort=True).getInfo()

new_spill_px  = sar_change.select("new_spill_mask").reduceRegion(
    reducer=ee.Reducer.sum(), geometry=ROI, scale=100, maxPixels=1e9, bestEffort=True).getInfo()
chronic_px = sar_change.select("chronic_spill_mask").reduceRegion(
    reducer=ee.Reducer.sum(), geometry=ROI, scale=100, maxPixels=1e9, bestEffort=True).getInfo()

print("SAR change mean   : " + str(round(change_stats.get("SAR_change_dB_mean", 0) or 0, 2)) + " dB")
print("New spill pixels  : " + str(int(new_spill_px.get("new_spill_mask", 0) or 0)))
print("Chronic spill px  : " + str(int(chronic_px.get("chronic_spill_mask", 0) or 0)))


=== SAR Change Detection ===
  S1 scenes (2023-01-01 to 2023-06-30): 30
  S1 scenes (2024-01-01 to 2024-06-30): 29
SAR change mean   : -0.01 dB
New spill pixels  : 1377
Chronic spill px  : 11685


## Cell 7 — DBSCAN clustering

In [7]:
print("=== DBSCAN Clustering ===")

if df_firms.empty:
    df_clusters = pd.DataFrame()
    print("No FIRMS data.")
else:
    coords  = df_firms[["lat","lon"]].values
    eps_rad = np.radians(0.05)
    db      = DBSCAN(eps=eps_rad, min_samples=2, algorithm="ball_tree",
                     metric="haversine").fit(np.radians(coords))
    df_firms["cluster"] = db.labels_

    n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    n_noise    = (db.labels_ == -1).sum()
    print("Hotspots  : " + str(len(df_firms)))
    print("Clusters  : " + str(n_clusters))
    print("Noise pts : " + str(n_noise))
    print()

    summary = []
    for cid in sorted(set(db.labels_)):
        if cid == -1: continue
        g   = df_firms[df_firms["cluster"] == cid]
        src = g["likely_source"].value_counts().index[0]
        risk = "HIGH" if src == "refinery_candidate" else "MEDIUM" if src == "flare_candidate" else "LOW"
        r = {"cluster_id": cid, "n_hotspots": len(g),
             "centroid_lat": g["lat"].mean(), "centroid_lon": g["lon"].mean(),
             "mean_T21_K": g["T21_max_K"].mean(), "max_T21_K": g["T21_max_K"].max(),
             "mean_fire_count": g["fire_count"].mean(),
             "dominant_source": src, "risk_label": risk}
        summary.append(r)
        print("  C" + str(cid) + ": " + str(len(g)) + " hotspots"
              + "  lat=" + str(round(r["centroid_lat"],3))
              + "  lon=" + str(round(r["centroid_lon"],3))
              + "  T21_max=" + str(round(r["max_T21_K"],0)) + "K"
              + "  risk=" + risk)

    df_clusters = pd.DataFrame(summary)
    print()
    print("Cluster table: " + str(len(df_clusters)) + " sites")


=== DBSCAN Clustering ===
Hotspots  : 50
Clusters  : 8
Noise pts : 7

  C0: 2 hotspots  lat=5.089  lon=6.513  T21_max=349.0K  risk=LOW
  C1: 12 hotspots  lat=5.715  lon=6.528  T21_max=349.0K  risk=LOW
  C2: 2 hotspots  lat=5.282  lon=6.813  T21_max=341.0K  risk=HIGH
  C3: 3 hotspots  lat=5.561  lon=6.876  T21_max=332.0K  risk=LOW
  C4: 2 hotspots  lat=5.038  lon=6.948  T21_max=342.0K  risk=LOW
  C5: 4 hotspots  lat=5.307  lon=7.007  T21_max=346.0K  risk=LOW
  C6: 3 hotspots  lat=5.24  lon=7.069  T21_max=338.0K  risk=LOW
  C7: 15 hotspots  lat=5.187  lon=7.155  T21_max=347.0K  risk=LOW

Cluster table: 8 sites


## Cell 8 — DBSCAN scatter plot (saved to outputs/)

In [9]:
import plotly.graph_objects as go

fig = go.Figure()

unique_clusters = sorted(df_firms['cluster'].unique())
colors = ['#888780','#E24B4A','#EF9F27','#378ADD','#1D9E75',
          '#7F77DD','#D85A30','#D4537E','#639922','#BA7517']

for i, cid in enumerate(unique_clusters):
    mask  = df_firms['cluster'] == cid
    g     = df_firms[mask]
    color = '#888780' if cid == -1 else colors[i % len(colors)]
    name  = 'Noise' if cid == -1 else 'Cluster ' + str(cid)
    fig.add_trace(go.Scatter(
        x=g['lon'], y=g['lat'],
        mode='markers',
        name=name,
        marker=dict(
            color=color,
            size=(g['T21_max_K'] - 300).clip(lower=5) / 3,
            opacity=0.8,
            line=dict(color='white', width=0.5)
        ),
        text=['T21=' + str(round(t,1)) + 'K  count=' + str(round(c,1))
              for t, c in zip(g['T21_max_K'], g['fire_count'])],
        hovertemplate='%{text}<extra>' + name + '</extra>'
    ))

if len(df_clusters) > 0:
    fig.add_trace(go.Scatter(
        x=df_clusters['centroid_lon'],
        y=df_clusters['centroid_lat'],
        mode='markers+text',
        name='Centroids',
        marker=dict(symbol='star', size=14, color='white',
                    line=dict(color='black', width=1)),
        text=['C' + str(int(r['cluster_id'])) + ' ' + r['risk_label']
              for _, r in df_clusters.iterrows()],
        textposition='top right',
        textfont=dict(size=9)
    ))

fig.update_layout(
    title='PipelineWatch-NG - DBSCAN Fire Hotspot Clustering<br>Trans Niger Pipeline corridor',
    xaxis_title='Longitude',
    yaxis_title='Latitude',
    height=550,
    legend=dict(x=1.01, y=1)
)

fig.write_html(os.path.join(OUTPUT_DIR, 'm2_dbscan_clusters.html'))
print('Saved: outputs/m2_dbscan_clusters.html')
from IPython.display import IFrame
display(IFrame(src='../outputs/m2_dbscan_clusters.html', width='100%', height=580))

Saved: outputs/m2_dbscan_clusters.html


## Cell 9 — NDVI change chart

In [12]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('NDVI baseline vs recent',
                                    'Fire hotspot locations by cluster'))

# Left: NDVI bar chart
categories = ['Baseline mean', 'Recent mean']
values     = [base_mean, rec_mean]
colors     = ['#378ADD', '#E24B4A' if delta < 0 else '#1D9E75']

fig.add_trace(go.Bar(
    x=categories, y=values,
    marker_color=colors, opacity=0.85,
    text=[str(round(v, 3)) for v in values],
    textposition='outside',
    name='NDVI'
), row=1, col=1)

fig.add_hline(y=0.3, line_dash='dash', line_color='#854F0B',
              annotation_text='Degraded threshold (0.3)',
              row=1, col=1)

# Right: fire hotspot scatter coloured by cluster
if not df_firms.empty and 'cluster' in df_firms.columns:
    fig.add_trace(go.Scatter(
        x=df_firms['lon'], y=df_firms['lat'],
        mode='markers',
        marker=dict(
            color=df_firms['cluster'],
            colorscale='turbo',
            size=8, opacity=0.75,
            line=dict(color='white', width=0.3)
        ),
        text=['C' + str(c) + '  T21=' + str(round(t, 1)) + 'K'
              for c, t in zip(df_firms['cluster'], df_firms['T21_max_K'])],
        hovertemplate='%{text}<extra></extra>',
        name='Hotspots'
    ), row=1, col=2)

fig.update_layout(
    title='PipelineWatch-NG - Sentinel-2 Vegetation Analysis',
    height=480,
    showlegend=False
)

fig.update_yaxes(title_text='Mean NDVI', row=1, col=1)
fig.update_xaxes(title_text='Longitude', row=1, col=2)
fig.update_yaxes(title_text='Latitude', row=1, col=2)

delta_text = 'Change: ' + str(round(delta, 3)) + '  ->  ' + direction
fig.add_annotation(text=delta_text, xref='x domain', yref='y domain',
                   x=0.5, y=0.05, showarrow=False, row=1, col=1,
                   bgcolor='#FAEEDA', bordercolor='#854F0B')

fig.write_html(os.path.join(OUTPUT_DIR, 'm2_ndvi_analysis.html'))
print('Saved: outputs/m2_ndvi_analysis.html')
from IPython.display import IFrame
display(IFrame(src='../outputs/m2_ndvi_analysis.html', width='100%', height=510))

Saved: outputs/m2_ndvi_analysis.html


## Cell 10 — Feature table and export

In [13]:
print("=== Building Feature Table ===")

feature_stack = s1_recent.select(["VV","VH","dark_spot_mask","dark_spot_magnitude"])
if ndvi_anomaly:
    feature_stack = feature_stack.addBands(s2_recent.select(["NDVI","NDWI","NDMI"]))
    feature_stack = feature_stack.addBands(ndvi_anomaly.select("NDVI_change"))
feature_stack = feature_stack.addBands(
    sar_change.select(["SAR_change_dB","new_spill_mask","chronic_spill_mask"]))

gc.collect()
sampled     = feature_stack.sample(region=ROI, scale=5000, numPixels=300, geometries=True, seed=42)
sample_info = sampled.getInfo()
rows = []
for feat in sample_info["features"]:
    coords = feat["geometry"]["coordinates"]
    props  = feat["properties"]
    props["lon"] = coords[0]; props["lat"] = coords[1]
    rows.append(props)
df_features = pd.DataFrame(rows)
print("Feature table: " + str(len(df_features)) + " rows x " + str(len(df_features.columns)) + " cols")

feat_csv = os.path.join(CACHE_DIR, "m2_feature_table.csv")
df_features.to_csv(feat_csv, index=False)
df_firms.to_csv(os.path.join(CACHE_DIR, "m2_hotspots_clustered.csv"), index=False)
if len(df_clusters) > 0:
    df_clusters.to_csv(os.path.join(CACHE_DIR, "m2_refinery_clusters.csv"), index=False)

meta = {"module": "M2_processing", "exported": datetime.now().isoformat(),
        "feature_rows": len(df_features), "fire_clusters": len(df_clusters) if len(df_clusters) > 0 else 0}
with open(os.path.join(CACHE_DIR, "m2_metadata.json"), "w") as f:
    json.dump(meta, f, indent=2)
print("All outputs written to data/cached/")


=== Building Feature Table ===
Feature table: 270 rows x 13 cols
All outputs written to data/cached/


## Module 2 complete

| Output | File |
|--------|------|
| DBSCAN cluster map | outputs/m2_dbscan_clusters.png |
| NDVI analysis chart | outputs/m2_ndvi_analysis.png |
| Feature table | data/cached/m2_feature_table.csv |
| Cluster summary | data/cached/m2_refinery_clusters.csv |

**Next:** Run Module 3